# Exploratory Data Analysis

Author: Justin Winkler
Date: September 17, 2026

The purpose of this notebook is to explore the Zillow and data center location datasets.

Effective exploration will satisfy the following criteria:

1. *Profiles* the data:
    - What size is the data?
    - What type(s) does each column contain?
    - What do the distributions of each variable look like?
2. Evaluates the *completeness* of the data:
    - Are there any missing values in the dataset? If so:
        - How many and where?
        - Why might those values be missing?
        - Based on the answer to the following question, what is the best way to handle the missing values?
3. Ensures the *accuracy* of the data:
    - Is the data consistent with other trusted sources?
4. Verifies the *consistency* of data:
    - Are similar measurements recorded in consistent units?
    - Are observations duplicated across datasets? If so, do they match?
5. Enforces the *integrity* of the data:
    - Are IDs unique? Are they consistent between datasets?
6. Documents the data's *lineage and provenance*:
    - Where did the data come from?
    - How has the data been transformed?

## **Import essential data processing utilities**

In [51]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## **Load the data**

First, define a local path to the `data` directory.

In [52]:
from pathlib import Path

data_dir = Path("data").resolve()

#### Load data center location information

Rows in `data_centers.csv` correspond to unique data center locations in the U.S. and abroad. The full dataset contains numerical fields describing the energy consumption and output of each data center, categorial fields containing tags that describe each data center's user and owner, and other data outside the scope of this project.

First, limit data centers to only US locations to match the Zillow dataset. Then, select only the columns pertaining to data center names (a unique identifier for this dataset) and their addresses.

In [53]:
raw_data_centers = pd.read_csv(data_dir / "raw" / "epoch_ai" / "data_centers.csv")
us_data_centers = raw_data_centers[raw_data_centers["Country"] == "United States"]
us_data_center_locations = us_data_centers[["Name", "Address"]]

#### Load data center construction timeline information

Rows in `data_center_timelines.csv` correspond to aggregations of data center construction update news headlines. Fields in this dataset include the name of the data center the construction update belongs to, a description of the update, a count of the number of operational buildings at the data center site at the time of the update, as well as information about the energy consumption and cost of the data center.

Select only the columns pertaining to data center names (a unique identifier for this dataset), the date of the headline aggregation, and the number of buildings that were operational at the time of the aggregation. Rename the "Data center" field to "Name" and the "Buildings operational" field to "BuildingsOperational" to standardize the naming scheme for columns. Standardize date-like fields by converting them to datetime.

In [54]:
data_center_timelines = pd.read_csv(data_dir / "raw" / "epoch_ai" / "data_center_timelines.csv")
data_center_timelines = data_center_timelines[["Data center", "Date", "Buildings operational"]]
data_center_timelines = data_center_timelines.rename(columns={'Data center': 'Name', "Buildings operational": "BuildingsOperational"})
data_center_timelines["Date"] = pd.to_datetime(data_center_timelines.Date)

#### Define the event of interest

Control whether to study the impact of the "first headline" about data center construction or the completion of the first data center's construction on surrounding home values.

In [55]:
from enum import Enum

class EventOfInterest(Enum):
    FIRST_HEADLINE = 1
    FIRST_OPERATIONAL = 2

EVENT_OF_INTEREST = EventOfInterest.FIRST_OPERATIONAL

if EVENT_OF_INTEREST == EventOfInterest.FIRST_OPERATIONAL:
    # If looking for first operational date, remove dates with no operational buildings
    data_center_timelines = data_center_timelines[data_center_timelines["BuildingsOperational"] != 0]

data_center_dates_of_interest = data_center_timelines.groupby(by="Name")["Date"].min()
data_center_dates_of_interest.name = "DateOfInterest (DOI)"
print(data_center_dates_of_interest)

Name
AWS Berwick                      2025-09-09
AWS New Albany                   2024-12-01
Alibaba Zhangbei                 2025-09-17
Amazon Madison Mega Site         2025-06-23
Amazon Ridgeland                 2026-05-15
                                    ...    
Start Campus Sines Data Campus   2026-04-03
Stream Phoenix                   2026-05-01
VNET Bayin Ulanqab               2025-07-01
Vantage TX1                      2026-02-15
xAI QTS Atlanta                  2025-02-20
Name: DateOfInterest (DOI), Length: 86, dtype: datetime64[ns]


#### Combine the data center datasets

Create a unified dataset by joining the data center DataFrames on their name and keeping all columns of the trimmed datasets.

Print the first five entries of the dataset for a preview of the data's structure.

In [56]:
# Join the two primary datasets
data_centers = pd.merge(us_data_center_locations, data_center_dates_of_interest, on="Name", how="outer")

# Extract the last 5-digit run of characters (preceded by whitespace) in the "Address" field to the "ZipCode" field.
data_centers["ZipCode"] = data_centers['Address'].str.extract(r'(?<=\s)(\d{5})(?!.*\d)')

# Manually map a few data centers to their zip, sourced using Google Maps.
data_centers["ZipCode"] = data_centers["ZipCode"].fillna(data_centers["Name"].map({
    "AWS New Albany": "43054",
    "Google The Dalles": "97058",
    "Meta Huntsville": "35810",
    "CoreWeave Chester VA": "23836",
    "Microsoft-Nebius New Jersey": "08361",
    "Stream Phoenix": "85338",
    "Amazon Ridgeland": "39157",
    "Amazon Madison Mega Site": "39046",
}))

# Drop any remaining rows with NaN zip codes
data_centers = data_centers.dropna(subset=['ZipCode'])

# If a ZipCode has multiple dates of interest because more than one data center belongs to it, choose the earliest
data_centers = (
    data_centers.sort_values("DateOfInterest (DOI)")
    .groupby("ZipCode", as_index=False)
    .agg({"Name": list, "Address": "first", "DateOfInterest (DOI)": "first"})
)

# Visualize the new data
print(data_centers.head())

  ZipCode                                           Name  \
0   08361                  [Microsoft-Nebius New Jersey]   
1   14012  [Core42 Lake Mariner, Anthropic Lake Mariner]   
2   18603                                  [AWS Berwick]   
3   20109                   [STACK Infrastructure NVA02]   
4   20136                               [Google Bristow]   

                                      Address DateOfInterest (DOI)  
0    3963 S Lincoln Ave, Vineland, New Jersey           2026-04-15  
1              7725 Lake Rd, Barker, NY 14012           2025-10-01  
2        1125 Electron Ave, Berwick, PA 18603           2025-09-09  
3       9590 Hornbaker Rd, Manassas, VA 20109           2024-10-01  
4  13001 Rollins Ford Road, Bristow, VA 20136           2024-04-29  


#### Load Zillow home estimates

Rows in this dataset correspond to unique U.S. zip codes. The first few columns of the dataset are categorical, specifying the city, county, state, and metro area that the zip code belongs to while the remainder of the columns are numeric and specify the average single-family residence (SFR) value estimate, if one exists, for each month in the period from January 2000 to July 2026.

Select only the columns pertaining to zip codes and monthly average SFR values. Rename the "RegionName" field to "ZipCode". Standardize date-like column names by converting them to datetime.

Print the first five entries of the dataset for a preview of the data's structure.

In [57]:
from datetime import datetime

# Read then concatenate the two zestimate datasets. There are two datasets to accomodate
# GitHub's file size limit.
zestimates1 = pd.read_csv(data_dir / "raw" / "zillow" / "zestimates_by_zip_1.csv")
zestimates2 = pd.read_csv(data_dir / "raw" / "zillow" / "zestimates_by_zip_2.csv")
zestimates = pd.concat([zestimates1, zestimates2], ignore_index=True)

# Split the data into two DataFrame objects:
#   - `zip_geo`: categorical features describing geographical information about zip codes
#   - `zestimates_by_zip`: a chronological breakdown of average home value zestimates by zip code
zestimates_by_zip = zestimates.filter(regex=r'RegionName|\d{4}-\d{2}-\d{2}')
zip_geo = zestimates.filter(regex=r'RegionName|State|Metro|CountyName|SizeRank')

# Rename "RegionName" columns to "ZipCode" and standardize zip codes to 5-character strings.
# Pad the front with 0s, if necessary, since they are originally stored as int64s.
zestimates_by_zip = zestimates_by_zip.rename(columns={"RegionName": "ZipCode"})
zip_geo = zip_geo.rename(columns={"RegionName": "ZipCode"})
zip_geo["ZipCode"] = zip_geo["ZipCode"].astype(str).str.zfill(5)

# Standardize date-like column names
zestimates_by_zip.columns = ["ZipCode", *pd.to_datetime(zestimates_by_zip.columns[1:])]

# Visualize the new data
print(zestimates_by_zip.head())

   ZipCode  2000-01-31 00:00:00  2000-02-29 00:00:00  2000-03-31 00:00:00  \
0    77494        212188.981819        212373.142689        212865.242558   
1     8701        113571.789919        114040.532783        114357.352733   
2    77449        105369.930306        105384.490172        105255.149979   
3    11368        171194.526575        172896.026224        174174.055641   
4    77084        105256.097011        105211.953674        105025.045819   

   2000-04-30 00:00:00  2000-05-31 00:00:00  2000-06-30 00:00:00  \
0        213859.178025        213894.730929        213739.535344   
1        115159.988256        115995.021610        116974.287103   
2        105245.255125        105293.299359        105488.782188   
3        176307.350060        177837.295188        179501.272127   
4        104939.654419        104914.252908        105054.918073   

   2000-07-31 00:00:00  2000-08-31 00:00:00  2000-09-30 00:00:00  ...  \
0        212973.219993        213006.239062        2127

#### Combine data center timeline and home value datasets

In [58]:
# DISCLAIMER: The following section was generated by Claude Code's Opus 5 model.

# -------------------------- GENERATED WITH HELP FROM CLAUDE CODE --------------------------

EVENT_WINDOW_MONTHS = 24

# Reshape the wide Zillow table into one row per zip code per month. Zillow stores zip
# codes as integers, so zero-pad them to match the five-character strings in data_centers.
zestimates_by_month = zestimates_by_zip.melt(id_vars="ZipCode", var_name="Date", value_name="Zestimate")
zestimates_by_month = zestimates_by_month.dropna(subset="Zestimate")
zestimates_by_month["ZipCode"] = zestimates_by_month["ZipCode"].astype(str).str.zfill(5)
zestimates_by_month["Date"] = pd.to_datetime(zestimates_by_month["Date"])
zestimates_by_month["LogValue"] = np.log(zestimates_by_month["Zestimate"])
zestimates_by_month = zestimates_by_month.sort_values("Date")

# Anchor each treated zip code to the last Zillow observation strictly before its date of
# interest. merge_asof requires both frames to be sorted on the matched date.
anchors = pd.merge_asof(
    data_centers[["ZipCode", "DateOfInterest (DOI)"]].sort_values("DateOfInterest (DOI)"),
    zestimates_by_month[["ZipCode", "Date"]],
    left_on="DateOfInterest (DOI)",
    right_on="Date",
    by="ZipCode",
    direction="backward",
    allow_exact_matches=False,
)
anchors = anchors.rename(columns={"Date": "AnchorDate"}).dropna(subset="AnchorDate")

# Restrict the panel to treated zip codes, express each observation's date in months
# relative to its anchor (EventTime = 0), and re-base log values to the anchor month.
treated_panel = zestimates_by_month.merge(anchors[["ZipCode", "AnchorDate"]], on="ZipCode")
treated_panel["EventTime"] = (
    12 * (treated_panel["Date"].dt.year - treated_panel["AnchorDate"].dt.year)
    + (treated_panel["Date"].dt.month - treated_panel["AnchorDate"].dt.month)
)
treated_panel = treated_panel[treated_panel["EventTime"].abs() <= EVENT_WINDOW_MONTHS].copy()
baseline = treated_panel["LogValue"].where(treated_panel["EventTime"] == 0)
treated_panel["RelLogValue"] = treated_panel["LogValue"] - baseline.groupby(treated_panel["ZipCode"]).transform("first")

# -------------------------- GENERATED WITH HELP FROM CLAUDE CODE --------------------------

treated_panel.to_csv(data_dir / "clean"  / "data_center_zipcode_value_estimates.csv")

In [ ]:
treated_zips = treated_panel["ZipCode"].unique().tolist()
treated_zip_geo = zip_geo[zip_geo["ZipCode"].isin(treated_zips)]
untreated_zip_geo = zip_geo[~zip_geo["ZipCode"].isin(treated_zips)]

# Donors come from the same metro, or the same state for treated zips without one.
has_metro = treated_zip_geo["Metro"].isna()
donor_pools = pd.concat([
    treated_zip_geo[has_metro].merge(untreated_zip_geo, on="Metro", suffixes=("", "Donor")),
    treated_zip_geo[~has_metro].merge(untreated_zip_geo, on="State", suffixes=("", "Donor")),
])[["ZipCode", "ZipCodeDonor"]].rename(columns={"ZipCode": "TreatedZip", "ZipCodeDonor": "DonorZip"})

print(donor_pools.groupby("TreatedZip").size())

TypeError: agg function failed [how->median,dtype->object]